# Efektywne programowanie w języku Python

## Wykład 3 — funkcje i leniwe przetwarzanie

### Jak korzystać z materiału?

Notebook obejmuje trzy bloki zgodne z planem zajęć:

1. **Funkcje i argumenty** — spotkanie 3.
2. **Iteratory i generatory** — spotkanie 4.
3. **Domknięcia i dekoratory** — spotkanie 5, z powrotem do zakresów nazw.

`part01` wprowadził funkcje, `part02` — kolekcje, a `opt02` — skutki uboczne i wartości domyślne. Tutaj łączymy te pojęcia. Przykłady działają w Pythonie 3.10+.

## A. Funkcje i argumenty

Cel: zaprojektować funkcję z jasnym wynikiem, zakresem nazw i sposobem przekazywania argumentów.

Najpierw określ kontrakt, potem sprawdź wynik, przypadki brzegowe i skutki uboczne. Komórki opisane jako „celowy błąd” służą odczytaniu wyjątków; uruchamiaj je świadomie, osobno od dalszych przykładów.

### Parametr, argument i wynik

**Parametr** to nazwa w definicji, a **argument** to wartość przekazana przy wywołaniu. Ciało zwykłej funkcji wykonuje się dopiero przy jej wywołaniu.

`return` kończy bieżące wywołanie i przekazuje obiekt wywołującemu. `print()` wypisuje tekst — to inny skutek niż zwracanie wyniku.

In [ ]:
def add(left, right):
    """Zwróć sumę dwóch wartości obsługujących dodawanie."""
    return left + right

result = add(2, 3)
print(result)
assert result == 5

### Brak return oznacza None

Dotarcie do końca zwykłej funkcji albo samo `return` daje `None`. Kilka wartości zapisanych po przecinkach jest pakowanych w jedną krotkę.

In [ ]:
def announce(message):
    print(message)

def bounds(values):
    return min(values), max(values)

assert announce("Witaj") is None
lowest, highest = bounds([4, 1, 9])
assert (lowest, highest) == (1, 9)
print(lowest, highest)

### Kontrakt obejmuje również błędy

`bounds()` przyjmuje niepustą sekwencję (np. listę), którą można odczytać wielokrotnie; w tym przykładzie dokumentujemy ten warunek zamiast wymyślać domyślny wynik. Wyjątki są częścią zachowania funkcji.

W rozbudowanych rekordach zamiast nieopisanej krotki można rozważyć `namedtuple` lub `dataclass` — wrócimy do nich przy obiektach.

In [ ]:
# Celowy ValueError: min() i max() nie mają wyniku dla pustego wejścia.
bounds([])

### Zakresy nazw: LEGB

Dla zwykłych odwołań wewnątrz funkcji szukamy nazwy kolejno w:

- **Local** — bieżącym wywołaniu;
- **Enclosing** — otaczających funkcjach;
- **Global** — module;
- **Builtins** — nazwach wbudowanych.

Przypisanie do nazwy wewnątrz funkcji zwykle czyni ją lokalną. Specjalne reguły klas omówimy przy klasach.

In [ ]:
scope_value = 2

def show_scope(argument):
    local_value = 5
    print(locals())
    print(scope_value, argument, local_value)

show_scope(3)  # Odczyt globalnej wartości jest dozwolony.

In [ ]:
def shadow_scope(argument):
    scope_value = 41
    print(scope_value, argument)
    print(globals()["scope_value"])  # Tylko demonstracja przestrzeni modułu.

shadow_scope(3)
assert scope_value == 2

### if i for nie tworzą osobnego zakresu

`if`, `for`, `while` i `with` nie wprowadzają nowego zakresu nazw. Nie oznacza to, że tylko `def` tworzy zakres: moduły i klasy mają własne przestrzenie nazw, a zmienna sterująca comprehension nie wycieka do otoczenia.

`locals()` służy tu do obserwacji. Nie używamy modyfikowania jego wyniku jako sposobu ustawiania zmiennych lokalnych.

In [ ]:
success = True
if success:
    description = "Winner!"
else:
    description = "Loser"
print(description)

number = 99
squares = [number * number for number in range(3)]
assert number == 99
print(squares, number)

### Dlaczego odczyt może zgłosić UnboundLocalError?

Przypisanie w dowolnym miejscu ciała funkcji powoduje traktowanie tej nazwy jako lokalnej w całej funkcji, jeśli nie użyto `global` ani `nonlocal`. Odczyt przed przypisaniem nie przechodzi wtedy do zmiennej globalnej.

In [ ]:
scope_count = 10

def broken_increment():
    scope_count += 1
    return scope_count

# Celowy błąd: lokalny scope_count nie ma jeszcze wartości.
broken_increment()

### global: jawna zmiana powiązania w module

`global` pozwala przypisać wartość do nazwy modułu. To mechanizm języka, ale funkcja przyjmująca wartość i zwracająca wynik zwykle łatwiej poddaje się testowaniu.

Zmianę nazwy z otaczającej funkcji realizuje `nonlocal` — pokażemy go przy domknięciach.

In [ ]:
def increment_global():
    global scope_count
    scope_count += 1

increment_global()
assert scope_count == 11
print(scope_count)

def increment(value):
    return value + 1

assert increment(10) == 11

### Argumenty wiążą parametry z obiektami

To ta sama zasada referencji co w `opt02`: mutacja przekazanej listy może być widoczna u wywołującego. Ponowne przypisanie parametru zmienia lokalne powiązanie.

Liczby i napisy nie mają osobnego mechanizmu „przekazywania przez wartość”. Są niemutowalne.

In [ ]:
def append_item(values):
    values.append(3)

def replace_items(values):
    values = [0]
    return values

items = [1, 2]
append_item(items)
replacement = replace_items(items)
assert items == [1, 2, 3] and replacement == [0]
print(items, replacement)

### Argumenty domyślne: przypomnienie z opt02

Wartość domyślna jest obliczana przy wykonaniu `def`, nie przy każdym wywołaniu. Gdy pominięty argument ma oznaczać nową listę, użyj `None` i utwórz ją wewnątrz funkcji.

Nie powtarzamy pełnej demonstracji wspólnej listy z `opt02`; skupiamy się na projektowaniu interfejsu.

In [ ]:
def collect(value, values=None):
    if values is None:
        values = []
    values.append(value)
    return values

assert collect(1) == [1]
assert collect(2) == [2]
provided = []
assert collect(3, provided) is provided
assert provided == [3]

### Argumenty pozycyjne i nazwane

Poniższa funkcja ilustruje różne sposoby wywołania. Parametr `species` nie zasłania funkcji wbudowanej `type`.

Ta sama wartość parametru nie może zostać przekazana jednocześnie pozycyjnie i przez nazwę.

In [ ]:
def parrot(voltage, state="a stiff", action="voom", species="Norwegian Blue"):
    print(f"This parrot would not {action} at {voltage} volts.")
    print(f"Lovely plumage, the {species}. State: {state}.")

### Poprawne wywołania

In [ ]:
parrot(1000)
parrot(voltage=1000)
parrot(action="VOOOOOM", voltage=1_000_000)

In [ ]:
parrot("a million", "bereft of life", "jump")
parrot("a thousand", state="pushing up the daisies")

### Błędy wywołania: jeden problem na komórkę

Brak obowiązkowego argumentu, podwójne przypisanie i nieznana nazwa parametru zgłaszają `TypeError`. Nieznane nazwy są akceptowane tylko wtedy, gdy sygnatura przewiduje `**kwargs`.

In [ ]:
# Celowy TypeError: brak voltage.
parrot()

In [ ]:
# Celowy TypeError: voltage podano dwukrotnie.
parrot(110, voltage=220)

In [ ]:
# Celowy TypeError: actor nie jest parametrem funkcji.
parrot(1000, actor="John Cleese")

### Błąd składni a błąd wiązania argumentów

Przykład ilustracyjny — nie uruchamiamy go jako kodu:

`parrot(voltage=5.0, "dead")` → `SyntaxError`.

Zwykły argument pozycyjny nie może wystąpić po argumencie nazwanym. Błędy składni są wykrywane przed wykonaniem komórki. Rozpakowywanie przez `*` ma dodatkowe reguły; nie używamy go tu do obchodzenia czytelnej kolejności.

### Zestawienie rodzajów parametrów

`def example(pos_only, /, normal, *args, named_only, **kwargs): ...`

- Przed `/`: tylko pozycyjnie (składnia dostępna od Pythona 3.8).
- `normal`: pozycyjnie lub przez nazwę.
- `*args`: nadmiarowe argumenty pozycyjne jako krotka.
- Po `*args` albo samym `*`: tylko przez nazwę.
- `**kwargs`: nadmiarowe argumenty nazwane jako słownik.

Wartość domyślna i sposób przekazywania to niezależne cechy parametru.

In [ ]:
def format_record(identifier, /, label="item", *, uppercase=False):
    result = f"{identifier}: {label}"
    return result.upper() if uppercase else result

assert format_record(7, label="python", uppercase=True) == "7: PYTHON"
print(format_record(7))

### *args i keyword-only na działającym przykładzie

Nie dodajemy osobnego `*` po `*nums`: parametry po nim już są keyword-only.

Kontrakt: iloczyn wszystkich liczb pomnożony przez `scale`. Pusty iloczyn wynosi 1, więc `product()` daje 1, a `product(scale=10)` daje 10.

In [ ]:
def product(*nums, scale=1):
    result = scale
    for number in nums:
        result *= number
    return result

assert product(3, 5) == 15
assert product(3, 4, 2) == 24
assert product(3, 5, scale=10) == 150
assert product() == 1 and product(scale=10) == 10
print(product(3, 5, scale=10))

### Rozpakowanie * przy wywołaniu

`*primes` przekazuje elementy jako osobne argumenty. Implementacja sprawdzania pierwszości jest kompletna, a testy obejmują liczby mniejsze od 2.

W zwykłym programie istnieje też gotowe `math.prod(iterable, start=...)`. Własna funkcja służy tu poznaniu sygnatur.

In [ ]:
from math import isqrt, prod

def is_prime(number):
    if number < 2:
        return False
    return all(number % divisor != 0
               for divisor in range(2, isqrt(number) + 1))

assert all(not is_prime(n) for n in (-3, 0, 1, 4, 9))
assert all(is_prime(n) for n in (2, 3, 5, 97))

In [ ]:
primes = [number for number in range(2, 100) if is_prime(number)]
assert primes[:5] == [2, 3, 5, 7, 11] and len(primes) == 25
assert product(*primes) == prod(primes)
assert product(*primes, scale=10) == prod(primes, start=10)
print(primes)
print(product(*primes))

### **kwargs i zachowanie kolejności

Nadmiarowe argumenty nazwane trafiają do słownika. Dla używanej wersji Pythona kolejność jest zgodna z kolejnością przekazania argumentów.

Nazwy `args` i `kwargs` są konwencją, a nie słowami kluczowymi. Poniżej używamy opisowej nazwy `speaker_info`.

In [ ]:
def print_quote(quote, **speaker_info):
    print(">", quote)
    for key, value in speaker_info.items():
        print(f"{key}: {value}")

print_quote("If music be the food of love, play on.",
            playwright="Shakespeare", act=1, scene=1, speaker="Duke Orsino")

### Rozpakowanie ** przy wywołaniu

Klucze rozpakowywanego mapowania muszą być napisami. Przekazane w ten sposób argumenty podlegają takim samym regułom wiązania jak zwykłe argumenty nazwane; duplikat parametru nadal jest błędem.

In [ ]:
info = {"sonnet": 18, "line": 1, "author": "Shakespeare"}
print_quote("Shall I compare thee to a summer's day?", **info)
assert list(info) == ["sonnet", "line", "author"]

### Przykład interfejsu: pytanie Y/N

Kontrakt: `Y` daje `True`, `N` daje `False`; ignorujemy spacje i wielkość liter. Po wyczerpaniu prób zgłaszamy `ValueError`, aby odróżnić brak poprawnej odpowiedzi od odmowy.

`retries` jest dodatnią liczbą całkowitą. Parametr keyword-only `read` pozwala podstawić źródło odpowiedzi w testach. Jeśli go pominiemy, funkcja korzysta z `input()`.

In [ ]:
def ask_yn(prompt, retries=4, complaint="Enter Y/N!", *, read=None):
    if type(retries) is not int or retries < 1:
        raise ValueError("retries musi być dodatnią liczbą całkowitą")
    if read is None:
        read = input
    for _ in range(retries):
        answer = read(prompt).strip().upper()
        if answer == "Y":
            return True
        if answer == "N":
            return False
        print(complaint)
    raise ValueError("Wyczerpano próby bez odpowiedzi Y/N")

### Test bez ręcznego wpisywania

Przekazujemy funkcję, która otrzymuje prompt i zwraca zaplanowany napis. Dzięki temu wykonanie całego notebooka nie czeka na użytkownika.

Wywołanie interaktywne `ask_yn("Really quit?")` pozostaje dostępne do samodzielnego sprawdzenia.

In [ ]:
answers = iter(["?", " y "])
accepted = ask_yn("Really quit?", retries=2, complaint="Tylko Y/N",
                  read=lambda prompt: next(answers))
assert accepted is True
assert ask_yn(prompt="Overwrite?", read=lambda prompt: "n") is False
assert ask_yn("Send?", complaint="Y/N please", read=lambda prompt: "Y") is True
print("Testy poprawnych odpowiedzi zakończone")

In [ ]:
# Celowy ValueError: dwie błędne odpowiedzi, a nie odmowa N.
ask_yn("Continue?", retries=2, read=lambda prompt: "?")

In [ ]:
# Celowy ValueError: nieprawidłowa konfiguracja liczby prób.
ask_yn("Continue?", retries=0, read=lambda prompt: "Y")

### Docstring opisuje kontrakt

Pierwszy literał napisu w ciele funkcji jest jej docstringiem. Krótko opisz cel, argumenty, wynik, skutki uboczne i istotne wyjątki.

Wybierz jeden spójny styl dokumentowania. Typy można też zapisać jako adnotacje; same adnotacje nie wymuszają sprawdzania typów podczas wykonania.

In [ ]:
def doubled(number: int) -> int:
    """Zwróć podwojoną liczbę bez zmiany argumentu.

    Kontrakt dydaktyczny: number jest liczbą całkowitą.
    Adnotacje dokumentują zamiar; ta funkcja nie waliduje typu.
    """
    return number * 2

assert doubled(3) == 6
print(doubled.__doc__)

### Ćwiczenie A: zaprojektuj funkcję

Napisz `scale_values(values, *, factor=1)`, która zwraca nową listę przeskalowanych liczb i nie zmienia wejścia.

1. Sprawdź pustą listę, współczynnik 0 i liczby ujemne.
2. Uzasadnij parametr keyword-only.
3. Dodaj docstring i asercje dotyczące wyniku oraz braku mutacji.

Do referencji i wartości domyślnych wróć w `opt02`, jeśli któryś test zaskakuje.

### Przykładowe rozwiązanie — po dyskusji

In [ ]:
def scale_values(values, *, factor=1):
    """Zwróć nową listę wartości pomnożonych przez factor."""
    return [value * factor for value in values]

source_values = [-2, 0, 3]
scaled = scale_values(source_values, factor=-2)
assert scaled == [4, 0, -6] and source_values == [-2, 0, 3]
assert scaled is not source_values
assert scale_values([], factor=2) == []
assert scale_values([1, 2], factor=0) == [0, 0]

## B. Iteratory i generatory

Cel: rozróżnić źródło danych od stanu iteracji, rozumieć zużycie iteratora i bezpiecznie korzystać z leniwych strumieni.

To rozwinięcie `range`, `zip` i wyrażeń generatorowych z `part02` oraz porównania czasu i pamięci z `opt01`.

### Iterable, iterator, generator

| Pojęcie | Co zapewnia? | Przykład |
|---|---|---|
| Obiekt iterowalny (iterable) | Można uzyskać iterator przez `iter()` | Lista, range, iterator |
| Iterator | `next()` daje kolejną wartość; `iter(it) is it` | `iter([1, 2])` |
| Generator | Szczególny iterator z zachowanym stanem wykonania | Wynik funkcji z `yield` |

Lista pozwala tworzyć kolejne niezależne iteratory. Przekazanie już istniejącego iteratora nie tworzy jego nowej, niezależnej kopii.

In [ ]:
data = [1, 2, 3]
first_iterator = iter(data)
second_iterator = iter(data)
assert first_iterator is not second_iterator
assert iter(first_iterator) is first_iterator
print(next(first_iterator), next(first_iterator))
print(next(second_iterator))  # Niezależny stan: nadal 1.

### Wyczerpanie iteratora

Po ostatnim elemencie `next(it)` zgłasza `StopIteration`. To sygnał zakończenia, który pętla `for` obsługuje automatycznie.

Wariant `next(it, default)` zwraca wartość domyślną zamiast tego wyjątku. Jeśli dane mogą zawierać tę samą wartość, wybierz jednoznaczny znacznik.

In [ ]:
iterator = iter([1, 2, 3])
print(next(iterator), next(iterator), next(iterator))
assert next(iterator, "koniec") == "koniec"
assert list(iterator) == []

In [ ]:
# Celowy StopIteration: iterator pozostaje wyczerpany.
next(iterator)

### Co robi for?

Schemat działania: pobiera iterator, odczytuje kolejne elementy, kończy po `StopIteration`.

Nie musimy ręcznie zapisywać tej obsługi. Poniżej obie pętle wykonują to samo dla świeżej listy; dla iteratora druga zaczęłaby od pozostałego stanu.

In [ ]:
data_source = [10, 20, 30]
for item in data_source:
    print(item, end=" ")
print()
for item in iter(data_source):
    print(item, end=" ")
print()

### Mutowanie źródła podczas iteracji

Nie ma ogólnej gwarancji, że każdy iterator bezpiecznie „zobaczy zmiany”. Iterator listy może odczytać zmieniony element, ale usuwanie elementów może powodować pomijanie danych.

Zmiana rozmiaru słownika lub zbioru podczas iteracji może zgłosić `RuntimeError`. Jeśli potrzebujesz stabilnego zestawu danych, rozważ kopię albo utworzenie nowej kolekcji.

In [ ]:
values = [1, 2, 3]
values_iterator = iter(values)
assert next(values_iterator) == 1
values[1] = 20
assert next(values_iterator) == 20  # Ten konkretny iterator listy.
print(list(values_iterator))

### Konsumenci iteratora

`max`, `min`, `sum` i `list` potrzebują całego skończonego wejścia. `any`, `all` oraz wyszukiwanie przez `in` mogą zakończyć się wcześniej, gdy wynik jest znany.

Te operacje zużywają przekazany iterator. Puste wejście: `any` daje `False`, `all` daje `True`; `max` i `min` bez wartości domyślnej zgłaszają `ValueError`.

In [ ]:
checks = iter([False, True, False])
assert any(checks) is True
assert list(checks) == [False]  # Pierwsze dwa elementy zostały zużyte.

checks = iter([True, False, True])
assert all(checks) is False
assert list(checks) == [True]
assert any([]) is False and all([]) is True
assert max([], default=None) is None

In [ ]:
search = iter([10, 20, 30])
assert 20 in search
assert list(search) == [30]
assert max(iter([1, 9, 3])) == 9
assert min(iter([1, 9, 3])) == 1

### Leniwe funkcje wbudowane

`enumerate`, `zip`, `map` i `filter` zwracają iteratory. `list(...)` konsumuje je i materializuje wynik.

Przy ponownym użyciu potrzebujesz nowego iteratora albo zapisanej kolekcji. Szczegóły `map` i `filter` rozwiniemy w `part04`.

In [ ]:
numbers = [1, 2, 3]
numbered = enumerate(numbers, start=1)
paired = zip(numbers, ["a", "b", "c"], strict=True)
mapped = map(doubled, numbers)
filtered = filter(lambda value: value % 2, numbers)
print(list(numbered), list(paired))
print(list(mapped), list(filtered))
assert list(mapped) == []

### Funkcja generatorowa a obiekt generatora

Wywołanie funkcji zawierającej `yield` tworzy generator. Ciało nie rusza od razu: zaczyna działać przy pierwszym pobraniu elementu.

`yield` zawiesza wykonanie, zachowując stan potrzebny do wznowienia. `return` lub koniec ciała kończy generator. Zwykłe funkcje też mogą pozostawić żywe obiekty, np. przez zwrócone domknięcie — nie wszystkie dane są automatycznie „wyrzucane”.

In [ ]:
def generate_ints(count):
    print("Start generatora")
    for number in range(count):
        yield number
    print("Koniec generatora")

generated = generate_ints(3)
print("Obiekt utworzony; ciało jeszcze nie działało")

In [ ]:
assert next(generated) == 0  # Dopiero tutaj pojawi się Start.
assert list(generated) == [1, 2]
assert list(generated) == []
print(list(generate_ints(0)))

In [ ]:
# Celowy StopIteration: generator został już zużyty.
next(generated)

### yield i return w jednym przykładzie

Funkcja z `yield` jest generatorowa także wtedy, gdy konkretne wywołanie zakończy się przed pierwszym `yield`. Samo `return` kończy strumień; nie dodaje elementu do wyniku.

In [ ]:
def until_negative(values):
    for value in values:
        if value < 0:
            return
        yield value

assert list(until_negative([1, 2, -1, 3])) == [1, 2]
assert list(until_negative([-1, 2])) == []
print(list(until_negative([0, 1, 2])))

### Nieskończony generator wymaga ograniczenia

Generator Fibonacciego nie ma końca. Pobieramy określoną liczbę elementów przez `islice()`; każde kolejne pobranie kontynuuje pracę od aktualnego stanu.

**Nie uruchamiaj `max(generate_fibs())` ani `list(generate_fibs())`** — nie otrzymasz końcowego wyniku. To pytanie do dyskusji, a nie wykonywalna demonstracja.

In [ ]:
from itertools import islice

def generate_fibs():
    left, right = 0, 1
    while True:
        left, right = right, left + right
        yield left

fibs = generate_fibs()
assert list(islice(fibs, 5)) == [1, 1, 2, 3, 5]
assert list(islice(fibs, 3)) == [8, 13, 21]
print(max(islice(generate_fibs(), 10)))  # Maksimum z 10 elementów.

### Wyrażenie generatorowe i materializacja

`(x * x for x in range(n))` tworzy generator; `[x * x for x in range(n)]` od razu tworzy listę. Generator może zmniejszyć potrzebę buforowania, ale nie gwarantuje szybszego wykonania.

W `opt01` mierzyliśmy czas i szczyt pamięci osobno. Leniwość pozwala także nie wykonywać pracy, której konsument nie potrzebuje.

In [ ]:
squared_stream = (number * number for number in range(5))
assert next(squared_stream) == 0
assert list(squared_stream) == [1, 4, 9, 16]
assert list(squared_stream) == []

### Rozszerzenie: yield from

Dla zwykłego przekazywania elementów skończonych kolekcji `yield from row` zastępuje pętlę `for value in row: yield value`.

Pełny protokół komunikacji z generatorami jest szerszy; tutaj używamy wyłącznie delegowania iteracji.

In [ ]:
def flatten(rows):
    for row in rows:
        yield from row

assert list(flatten([[1, 2], [], [3]])) == [1, 2, 3]
assert list(flatten([])) == []

### Ćwiczenie B: skończony fragment strumienia

Napisz `take_values(iterable, count)`, które zwraca listę co najwyżej `count` elementów i wymaga nieujemnej liczby całkowitej.

Sprawdź pustą listę, `count=0`, wejście krótsze od limitu i nieskończony generator. Dla `count=0` nie pobieraj żadnego elementu.

Użyj `islice()`; nie konwertuj całego wejścia na listę przed ograniczeniem.

### Przykładowe rozwiązanie — po dyskusji

In [ ]:
def take_values(iterable, count):
    """Zwróć co najwyżej count pierwszych elementów strumienia."""
    if type(count) is not int or count < 0:
        raise ValueError("count musi być nieujemną liczbą całkowitą")
    return list(islice(iterable, count))

assert take_values([], 3) == []
assert take_values([1, 2], 5) == [1, 2]
assert take_values(generate_fibs(), 5) == [1, 1, 2, 3, 5]
untouched = iter([7, 8])
assert take_values(untouched, 0) == [] and next(untouched) == 7

## C. Domknięcia i dekoratory

Cel: przekazać funkcję jako obiekt, zwrócić funkcję ze stanem oraz opakować wywołanie bez utraty wyniku i metadanych.

To wyjaśnia także przygotowane narzędzia z `opt01` i `opt02`, które przyjmowały funkcje do pomiaru.

### Funkcje jako argumenty

Przekazujemy obiekt funkcji bez nawiasów wywołania. `*args` i `**kwargs` pozwalają przekazać dalej jej argumenty.

Kontrakt poniżej: wykonaj funkcję dwa razy i zwróć oba wyniki. Skutki uboczne też wystąpią dwukrotnie.

In [ ]:
def perform_twice(function, *args, **kwargs):
    first = function(*args, **kwargs)
    second = function(*args, **kwargs)
    return first, second

assert perform_twice(product, 3, 5, scale=2) == (30, 30)
assert perform_twice(print, 5, 10, sep="&", end="...") == (None, None)
print()

### Funkcja jako wynik: domknięcie

Zwrócona funkcja korzysta z nazwy z otaczającego zakresu. Powiązanie pozostaje dostępne po zakończeniu wywołania funkcji zewnętrznej.

Kontrakt: dzielnik jest niezerową liczbą całkowitą. Walidujemy go podczas tworzenia testu, a nie dopiero przy pierwszym dzieleniu.

In [ ]:
def make_divisibility_test(divisor):
    if type(divisor) is not int or divisor == 0:
        raise ValueError("divisor musi być niezerową liczbą całkowitą")
    def divisible(number):
        return number % divisor == 0
    return divisible

div_by_3 = make_divisibility_test(3)
assert div_by_3(4) is False and div_by_3(9) is True
assert make_divisibility_test(5)(10) is True
print(div_by_3(12))

In [ ]:
# Celowy ValueError: błąd konfiguracji wykryty od razu.
make_divisibility_test(0)

### nonlocal: stan otaczającej funkcji

`nonlocal` pozwala przypisać wartość do już istniejącej nazwy z najbliższej odpowiedniej otaczającej funkcji. Nie odnosi się do zmiennej modułu.

Każde wywołanie fabryki poniżej tworzy niezależny stan licznika.

In [ ]:
def make_counter():
    count = 0
    def next_count():
        nonlocal count
        count += 1
        return count
    return next_count

counter_a = make_counter()
counter_b = make_counter()
assert (counter_a(), counter_a(), counter_b()) == (1, 2, 1)
print(counter_a())

### Dekorator opakowuje funkcję

Dekorator przyjmuje funkcję i zwraca obiekt, którym zastąpimy jej nazwę. Wrapper powinien przekazać argumenty i **zwrócić wynik** oryginału.

Na początek pokazujemy świadomie wersję bez zachowania metadanych, a następnie ją poprawimy.

In [ ]:
def debug_plain(function):
    def wrapper(*args, **kwargs):
        print("Arguments:", args, kwargs)
        return function(*args, **kwargs)
    return wrapper

def calculate_total(left, right, multiplier=1):
    """Zwróć sumę argumentów pomnożoną przez multiplier."""
    return (left + right) * multiplier

In [ ]:
plain_total = debug_plain(calculate_total)
assert plain_total(2, 3, multiplier=4) == 20
# Wypisuje: Arguments: (2, 3) {'multiplier': 4}
print(plain_total.__name__)  # wrapper
assert plain_total.__name__ == "wrapper"
assert plain_total.__doc__ is None

### functools.wraps zachowuje metadane

`@wraps(function)` zachowuje m.in. nazwę i docstring, a `__wrapped__` wskazuje opakowaną funkcję. Pomaga to w dokumentacji, diagnostyce i introspekcji.

`wraps` nie sprawdza poprawności argumentów ani nie naprawia logiki wrappera. Wciąż musimy przekazać argumenty i wynik.

In [ ]:
from functools import wraps

def debug(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("Arguments:", args, kwargs)
        return function(*args, **kwargs)
    return wrapper

wrapped_total = debug(calculate_total)
assert wrapped_total(2, 3, multiplier=4) == 20
assert wrapped_total.__name__ == calculate_total.__name__
assert wrapped_total.__doc__ == calculate_total.__doc__

### Składnia @debug

Zapis `@debug` nad definicją jest równoważny przypisaniu `name = debug(name)` po utworzeniu funkcji. Dekorator działa przy definiowaniu; wrapper działa podczas późniejszych wywołań.

In [ ]:
@debug
def decorated_total(left, right, multiplier=1):
    """Oblicz sumę pomnożoną przez multiplier."""
    return (left + right) * multiplier

assert decorated_total(5, 3, multiplier=2) == 16
assert decorated_total.__name__ == "decorated_total"
assert decorated_total.__wrapped__(5, 3, multiplier=2) == 16
print(decorated_total.__name__)

### Do czego używa się dekoratorów?

- Diagnostyka i pomiar czasu.
- Cache wyników, np. `functools.cache` — przy odpowiednim kontrakcie funkcji.
- Kontrola dostępu i routing w frameworkach.
- `property`, `staticmethod` i `classmethod` — wrócimy do nich przy klasach.

Dekorator sam z siebie nie gwarantuje przerwania blokującego kodu po timeout. Taki mechanizm wymaga dodatkowych narzędzi i założeń.

### Ćwiczenie C: dekorator sprawdzony testami

Napisz `count_calls`, który wypisuje numer wywołania i zwraca wynik funkcji. Zachowaj metadane przez `wraps`; do licznika użyj `nonlocal`.

Sprawdź wywołanie pozycyjne i nazwane, zwracany wynik oraz `__name__` i `__doc__`. Nie przechwytuj wyjątków funkcji bez ustalonej potrzeby.

### Przykładowe rozwiązanie — po dyskusji

In [ ]:
def count_calls(function):
    calls = 0
    @wraps(function)
    def wrapper(*args, **kwargs):
        nonlocal calls
        calls += 1
        print(f"Wywołanie {calls}")
        return function(*args, **kwargs)
    return wrapper

counted_total = count_calls(calculate_total)
assert counted_total(2, 3) == 5
assert counted_total(left=2, right=3, multiplier=4) == 20

In [ ]:
assert counted_total.__name__ == calculate_total.__name__
assert counted_total.__doc__ == calculate_total.__doc__
# Liczymy próby wywołania, również te zakończone błędem.
# Wyjątek oryginału nie powinien zostać zamieniony w None.
counted_total(2)  # Celowy TypeError: brak right.

## Podsumowanie — przewidź zachowanie

1. Czym różnią się `print()` i `return`?
2. Dlaczego przypisanie w funkcji może spowodować `UnboundLocalError`?
3. Co zbierają `*args` i `**kwargs`, a co robią `*` i `/`?
4. Co zostaje w iteratorze po `any()` i dlaczego trzeba ograniczać strumienie nieskończone?
5. Kiedy zaczyna działać ciało funkcji generatorowej?
6. Dlaczego poprawny dekorator powinien zachować wynik i metadane?

## Dokumentacja

- [Funkcje, parametry i rozpakowanie argumentów](https://docs.python.org/3.10/tutorial/controlflow.html#defining-functions).
- [Model wykonania: zakresy i wiązanie nazw](https://docs.python.org/3.10/reference/executionmodel.html).
- [Iterable, iterator i generator — słownik pojęć](https://docs.python.org/3.10/glossary.html#term-generator).
- [itertools.islice — ograniczanie strumienia](https://docs.python.org/3.10/library/itertools.html#itertools.islice).
- [functools.wraps — metadane dekoratorów](https://docs.python.org/3.10/library/functools.html#functools.wraps).

Dokumentacja wybranej wersji odpowiada środowisku kursu; przy pracy z inną wersją sprawdź jej dokumentację.

### Filmy uzupełniające

- [Brett Slatkin — How to Be More Effective with Functions, PyCon 2015](https://www.youtube.com/watch?v=WjJUPxKB164): argumenty zmienne ok. 2:30–6:00, keyword-only ok. 9:30–15:00, dalsza część jako rozszerzenie o generatorach.
- [Socratica — Decorators in Python](https://www.youtube.com/watch?v=WpF6azYAxYg).

Sprawdzone 2026-10-20: oba materiały otwierają się i odtwarzają. Są uzupełnieniem; definicje i kompletne przykłady znajdują się w notebooku i oficjalnej dokumentacji.